# 05. Tableau Dashboard Preparation

Tableau Dashboard에서 사용할 데이터셋을 생성한다.

04_KPI_Analysis에서 계산한 KPI 및 분석 결과를 CSV 파일로 저장하여 Tableau에서 시각화한다.

## 1. Connect Database

Tableau Dashboard에서 사용할 데이터를 생성하기 위해 SQLite 데이터베이스에 연결한다.

In [46]:
import pandas as pd
import sqlite3
from pathlib import Path

In [47]:
db_path = Path("../database/olist_dashboard.db")
conn = sqlite3.connect(db_path)

print("Database connected.")

Database connected.


## 2. Set Export Path

Tableau에서 사용할 CSV 파일을 저장할 경로를 설정한다.

`tableau/data` 폴더가 없으면 자동으로 생성한다.

In [49]:
export_path = Path("../tableau/data")
export_path.mkdir(parents=True, exist_ok=True)

print(f"Export Path: {export_path.resolve()}")

Export Path: C:\Users\USER\DataAnalysis\olist_dashboard\tableau\data


## 3. Monthly Performance

월별 KPI(매출, 주문 수, 고객 수, 평균 주문금액)를 계산하여 Tableau Dashboard에서 사용할 데이터셋을 생성한다.

In [51]:
monthly_performance = pd.read_sql("""
SELECT
    strftime('%Y-%m', o.order_purchase_timestamp) AS order_month,
    SUM(oi.price) AS product_sales,
    COUNT(DISTINCT o.order_id) AS total_orders,
    COUNT(DISTINCT c.customer_unique_id) AS total_customers
FROM orders AS o
JOIN order_items AS oi
    ON o.order_id = oi.order_id
JOIN customers AS c
    ON o.customer_id = c.customer_id
WHERE o.order_status = 'delivered'
  AND o.order_purchase_timestamp IS NOT NULL
GROUP BY order_month
ORDER BY order_month;
""", conn)

monthly_performance.head()

,order_month,product_sales,total_orders,total_customers
0,2016-09,134.97,1,1
1,2016-10,40325.11,265,262
2,2016-12,10.90,1,1
3,2017-01,111798.36,750,718
4,2017-02,234223.40,1653,1630


In [52]:
monthly_performance["aov"] = (
    monthly_performance["product_sales"]
    / monthly_performance["total_orders"]
).round(2)

monthly_performance.head()

,order_month,product_sales,total_orders,total_customers,aov
0,2016-09,134.97,1,1,134.97
1,2016-10,40325.11,265,262,152.17
2,2016-12,10.90,1,1,10.90
3,2017-01,111798.36,750,718,149.06
4,2017-02,234223.40,1653,1630,141.70


### Export Monthly Performance

월별 KPI 데이터셋을 CSV 파일로 저장하여 Tableau에서 사용한다.

In [54]:
monthly_performance.to_csv(
    export_path / "monthly_performance.csv",
    index=False
)

print("monthly_performance.csv saved.")

monthly_performance.csv saved.


## 4. Category Performance

카테고리별 KPI(매출, 주문 수, 고객 수)를 계산하여 Tableau Dashboard에서 사용할 데이터셋을 생성한다.

In [56]:
category_performance = pd.read_sql("""
SELECT
    COALESCE(p.product_category_name,'Unknown') AS product_category_name,
    SUM(oi.price) AS product_sales,
    COUNT(DISTINCT o.order_id) AS total_orders,
    COUNT(DISTINCT c.customer_unique_id) AS total_customers
FROM orders AS o
JOIN order_items AS oi
    ON o.order_id = oi.order_id
JOIN products AS p
    ON oi.product_id = p.product_id
JOIN customers AS c
    ON o.customer_id = c.customer_id
WHERE o.order_status = 'delivered'
GROUP BY COALESCE(
    p.product_category_name,
    'Unknown'
)
ORDER BY product_sales DESC;
""", conn)

category_performance.head()

,product_category_name,product_sales,total_orders,total_customers
0,beleza_saude,1233131.72,8647,8498
1,relogios_presentes,1166176.98,5495,5421
2,cama_mesa_banho,1023434.76,9272,9008
3,esporte_lazer,954852.55,7530,7341
4,informatica_acessorios,888724.61,6530,6405


In [57]:
category_performance["aov"] = (
    category_performance["product_sales"]
    / category_performance["total_orders"]
).round(2)

category_performance.head()

,product_category_name,product_sales,total_orders,total_customers,aov
0,beleza_saude,1233131.72,8647,8498,142.61
1,relogios_presentes,1166176.98,5495,5421,212.23
2,cama_mesa_banho,1023434.76,9272,9008,110.38
3,esporte_lazer,954852.55,7530,7341,126.81
4,informatica_acessorios,888724.61,6530,6405,136.10


### Export Category Performance

Tableau Dashboard에서 사용할 카테고리별 KPI 데이터셋을 CSV 파일로 저장한다.

In [59]:
category_performance.to_csv(
    export_path / "category_performance.csv",
    index=False
)

print("category_performance.csv saved.")

category_performance.csv saved.


## 5. Regional Performance

지역별 KPI(매출, 주문 수, 고객 수)를 계산하여 Tableau Dashboard에서 사용할 데이터셋을 생성한다.

In [61]:
regional_performance = pd.read_sql("""
SELECT
    c.customer_state,
    SUM(oi.price) AS product_sales,
    COUNT(DISTINCT o.order_id) AS total_orders,
    COUNT(DISTINCT c.customer_unique_id) AS total_customers
FROM orders AS o
JOIN order_items AS oi
    ON o.order_id = oi.order_id
JOIN customers AS c
    ON o.customer_id = c.customer_id
WHERE o.order_status = 'delivered'
GROUP BY c.customer_state
ORDER BY product_sales DESC;
""", conn)

regional_performance.head()

,customer_state,product_sales,total_orders,total_customers
0,SP,5067633.16,40501,39156
1,RJ,1759651.13,12350,11917
2,MG,1552481.83,11354,11001
3,RS,728897.47,5345,5168
4,PR,666063.51,4923,4769


In [62]:
regional_performance["aov"] = (
    regional_performance["product_sales"]
    / regional_performance["total_orders"]
).round(2)

regional_performance["sales_per_customer"] = (
    regional_performance["product_sales"]
    / regional_performance["total_customers"]
).round(2)

regional_performance.head()

,customer_state,product_sales,total_orders,total_customers,aov,sales_per_customer
0,SP,5067633.16,40501,39156,125.12,129.42
1,RJ,1759651.13,12350,11917,142.48,147.66
2,MG,1552481.83,11354,11001,136.73,141.12
3,RS,728897.47,5345,5168,136.37,141.04
4,PR,666063.51,4923,4769,135.30,139.67


### Export Regional Performance

Tableau Dashboard에서 사용할 지역별 KPI 데이터셋을
CSV 파일로 저장한다.

In [64]:
regional_performance.to_csv(
    export_path / "regional_performance.csv",
    index=False
)

print("regional_performance.csv saved.")

regional_performance.csv saved.


## 6. Delivery Performance

배송 관련 KPI(평균 배송 기간, 배송 건수, 정시 배송률)를 계산하여 Tableau Dashboard에서 사용할 데이터셋을 생성한다.

In [66]:
delivery_performance = pd.read_sql_query("""
SELECT
    c.customer_state,
    ROUND(
        AVG(
            julianday(o.order_delivered_customer_date)
            - julianday(o.order_purchase_timestamp)
        ),
        2
    ) AS avg_delivery_days,
    COUNT(DISTINCT o.order_id) AS total_deliveries,
    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN o.order_delivered_customer_date
                     <= o.order_estimated_delivery_date
                THEN 1
                ELSE 0
            END
        )
        / COUNT(DISTINCT o.order_id),
        2
    ) AS on_time_rate
FROM customers c
JOIN orders o
ON c.customer_id = o.customer_id
WHERE o.order_status = 'delivered'
GROUP BY c.customer_state
ORDER BY total_deliveries DESC;
""", conn)

delivery_performance

,customer_state,avg_delivery_days,total_deliveries,on_time_rate
0,SP,8.76,40501,94.09
1,RJ,15.31,12350,86.53
2,MG,12.01,11354,94.39
3,RS,15.30,5345,92.83
4,PR,11.99,4923,95.00
5,SC,14.95,3546,90.24
6,BA,19.34,3256,85.96
7,DF,12.97,2080,92.93
8,ES,15.79,1995,87.77
9,GO,15.61,1957,91.82


In [67]:
delivery_performance.to_csv(
    export_path / "delivery_performance.csv",
    index=False
)

print("delivery_performance.csv saved.")

delivery_performance.csv saved.
